In [33]:
# Sam Brown
# sam_brown@mines.edu
# June 9, 2025
# Goal: Form and save a dataframe that will be used to train a neural net
# NOTE: need to standardize values for stations
import my_lib.funcs
import Stations

# tide calcs (move to library eventually)
import Tides
import util.coordinate_transforms
import pyTMD
import datetime
import time
import scipy

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# One neural net/ dataset for events averaged out and one dataset for individual instances of stations



In [35]:
# first dataset: gz stations: 2011 to 2013
# DataSet can be used to predict most of the columns. We are interested in how long it will take until the next event. 
# When using to predict how long until the next event, we will offset the time_since column to be minutes until next event
# columns = Slip impulsivity, time since last event, slip length, tide height, form factor, tide derivative, slip pre-area, impulsive_class
                                
# Load in Raw Data
events_list2011 = my_lib.funcs.load_evt("/Users/sambrown04/Documents/SURF/Events/2011_2011Events2stas")
events_list2012 = my_lib.funcs.load_evt("/Users/sambrown04/Documents/SURF/Events/2012_2012Events2stas")
events_list2013 = my_lib.funcs.load_evt("/Users/sambrown04/Documents/SURF/Events/2013_2013Events2stas")


# Use preprocessing function to get some of the features
pre_11 = my_lib.funcs.extract_event_features(events_list2011)
pre_12 = my_lib.funcs.extract_event_features(events_list2012)
pre_13 = my_lib.funcs.extract_event_features(events_list2013)

# Merge into one large list of Dataframes
tot_dat = pre_11 + pre_12 + pre_13

# Keep only Gz stations
for i in range(len(tot_dat)):
    gz_rows = tot_dat[i][tot_dat[i]['station'].str.contains('gz')]
    # if gz rows exist, pass them
    if len(gz_rows) > 0:
        tot_dat[i] = gz_rows
    else: # If no gz rows exist (only 15 instances), pass a row with only start_time
        
        first_row = tot_dat[i].iloc[0].copy()
        
        # Set all columns to NaN except 'start_time'
        cols_to_nan = first_row.index.difference(['start_time'])
        first_row[cols_to_nan] = np.nan
        
        # 1-row DataFrame again
        tot_dat[i] = pd.DataFrame([first_row])
    
# Define df
avg_dat = pd.DataFrame(columns = ['pre_slip_area', 'total_delta', 'start_time', 'slip_severity', 'tide_height', 'tide_change', 'form_factor', 'mins_since'])
count = 0
# Loop to collect averages
for event in tot_dat:
    avg_dat.loc[len(avg_dat)] = {
        "pre_slip_area": event['pre-slip_area'].mean(),
        "total_delta": event['total_delta'].mean(),
        "start_time": event.iloc[0].loc['start_time'], # Start time is same for all so just take first station's
        "slip_severity": event['slip_severity'].mean()
    }
    

# Organize by time
avg_dat['start_time'] = pd.to_datetime(avg_dat['start_time'], errors='coerce')
avg_dat = avg_dat.sort_values('start_time')

# First 15 events don't have gz stations so we will remove them
avg_dat = avg_dat.iloc[15:]

# Calculate minutes since last event
avg_dat['mins_since'] = avg_dat['start_time'].diff().dt.total_seconds() / 60

# Retrieve Tide Height. We will use average coors of all gz stations for tide model. Code for how average coordinates are retrieved in severity_classification
x_cor = -168955.1491394913 
y_cor = -599694.5432784811

tide_df = get_tide_height(1100, x_cor, y_cor, "2011-01-01 00:00:00") # tide height is in centimeters (1100 days = 3 years)

# Only down to minutes, 
tide_df['time'] = tide_df['time'].apply(lambda x: x.strftime("%Y-%m-%d %H:%M"))
avg_dat['start_time'] = avg_dat['start_time'].apply(lambda x: x.strftime("%Y-%m-%d %H:%M"))


# Put corresponding tide height into our main dataframe
for i, row in avg_dat.iterrows():
    # Identify start time for row
    time_str = row['start_time']

    # Find the index that has this time in the tide data
    index = tide_df[tide_df['time'] == time_str].index

    # Insert time into our df
    if not index.empty:
        tide_r = index[0]
        avg_dat.at[i, 'tide_height'] = tide_df.at[tide_r, 'tide_height']

# Insert Tide Derivatives into our dataset
tide_d = tide_derivative(tide_df)

for i, row in avg_dat.iterrows():
    time = row['start_time']

    index = tide_d[tide_d['time'] == time].index

    if not index.empty:
        idx = index[0]
        avg_dat.at[i, 'tide_change'] = tide_d.at[idx, 'tide_deriv']



Elapsed time: 27.96634817123413 seconds


NameError: name 'df' is not defined

In [37]:
# Add new feature to be minutes until next event to predict how long until the next event
avg_dat['mins_until'] = avg_dat['mins_since'].shift(-1)

In [43]:
avg_dat.to_csv('averages_events_2011-13', index = False)

In [7]:
def get_tide_height(days, x_cor, y_cor, start_time):
    """ Get tide height for <days> days from initial date <start_time>, at the coordinates <x_cor> and <y_cor>.

    Parameters
    ----------
    days : int
        Number of days to calculate tide height for.
    x_cor : float
        PS71 x coordinate of tide calculation
    y_cor : _type_
        PS71 y coordinate of tide calculation
    start_time : _type_
        Starting date in %Y-%m-%d %H:%M:%S format

    Returns
    -------
    list[float]
        Tide heights
    """

    ### USER DEFINED PATH TO TIDE MODEL ###
    tide_dir = "/Users/sambrown04/Documents/SURF"
    #######################################
    
    tide_mod = "CATS2008-v2023"
    
    tides = Tides.Tide(tide_mod, tide_dir)
    
    spacing = 1 # every minute

    HR_PER_DAY = 24
    MIN_PER_HR = 60

    dates_timeseries = []
    initial_time = datetime.datetime.strptime(start_time, "%Y-%m-%d %H:%M:%S")
    for i in range(days * HR_PER_DAY * MIN_PER_HR // spacing):  # 30 days * 24 hr/day * 60 min/hr * 1/10 calculations/min
        dates_timeseries.append(initial_time + datetime.timedelta(minutes=spacing * i))

    #convert to lon and lat
    lon, lat = util.coordinate_transforms.xy2ll(x_cor, y_cor)
    # print(lon, lat)

    
    tides = Tides.Tide(tide_mod, tide_dir)
    
    start_time = time.time()
    tide_results = tides.tidal_elevation(
        [lon],
        [lat],
        dates_timeseries,
    ).data.T[0]
    end_time = time.time()
    elapsed_time = end_time - start_time
    print(f"Elapsed time: {elapsed_time} seconds")

    
    out = pd.DataFrame(columns = ["time", "tide_height"])
    out.loc[:,"time"] = dates_timeseries
    out.loc[:,"tide_height"] = tide_results

    return out
    

In [9]:
def tide_derivative(tide_df):
    """
    Calculates the derivative of a tide dataset

    Parameters
    ----------
    tide_df: pd.DataFrame
        DataFrame with two columns = ['time', 'tide_height']

    returns: pd.DataFrame
        columns = ['time', 'tide_derivative'] 
        1st derivative will be in units of cm/minute

    NOTE: Time should increment by minutes
    """
    
    # Retrieve our "f" values
    f = tide_df['tide_height'].to_numpy()

    # Paramter 1 signifies there is one minute between tide measurements
    # Since equal distances, this is a standard 2nd-order approx. under the hood.
    deriv = np.gradient(f, 1) 

    # Define our output df
    out = pd.DataFrame(columns = ['time', 'tide_deriv'])
    out['time'] = tide_df['time']

    out['tide_deriv'] = pd.Series(deriv)

    return out

In [10]:
def form_factor_calc(tide_time, days = 3, slide = 1):
    """
    Calculates form factor for tide data

    Parameters
    ----------
    tide_time: pd.DataFrame
        columns = ['time', 'tide_height']

    days: int
        number of days to perform tide calculation on

    slide: int
        calculation parameter

    Returns
    -------
    out: DataFrame
        columns = ['date', 'form_factors']
    """
    reference_time = tide_time['time'].iloc[0]
    seconds = [(date - reference_time).total_seconds() for date in tide_time['time']]
    
    # print(tide_time['tide_height'].shape, tide_time['time'].shape)
    
    tide = tide_time['tide_height']
    dates_timeseries = tide_time['time']
    
    spacing = 4  # Minutes
    mean_days = days
    slide_days = slide
    mean_units = int(mean_days * 24 * 60 / spacing)
    slide_units = int(slide_days * 24 * 60 / spacing)
    
    HR_TO_SEC = 3600
    T_O1 = 25.81933871 * HR_TO_SEC
    T_K1 = 23.93447213 * HR_TO_SEC
    T_M2 = 12.4206012 * HR_TO_SEC
    T_S2 = 12 * HR_TO_SEC
    
    def sines(x, A1, phi1, A2, phi2):
        return A1 * np.sin(2 * np.pi * x / ((T_O1 + T_K1) / 2) + phi1) + A2 * np.sin(
            2 * np.pi * x / ((T_M2 + T_S2) / 2) + phi2
        )
    
    form_factors = []
    dates_form_factor = []
    semidiurnal = []
    diurnal = []
    
    start = 0
    end = mean_units
    while end < len(seconds):
        seconds_tide = np.array(seconds[start:end], dtype=float)
        tide_window = np.array(tide[start:end], dtype=float)
        date_midpoint = dates_timeseries[(start + end) // 2]
        start += slide_units
        end += slide_units
    
        # Fit a sum of sines to the tide
        initial_guess = [50, 0, 50, 0]
        popt, pcov = scipy.optimize.curve_fit(sines, seconds_tide, tide_window, p0=initial_guess)
    
        # Extract fitted parameters
        Diurnal_fit, phi1_fit, SemiDiurnal_fit, phi2_fit = popt
    
        # Generate the fitted curve
        y_fit = sines(seconds_tide, Diurnal_fit, phi1_fit, SemiDiurnal_fit, phi2_fit)
        form_factor = np.abs(Diurnal_fit / SemiDiurnal_fit)
        semidiurnal.append((SemiDiurnal_fit))
        diurnal.append((Diurnal_fit))
    
        form_factors.append(form_factor)
        dates_form_factor.append(date_midpoint)

    # DataFrame to return
    out = pd.DataFrame(columns = ['dates', 'form_factors'])
    out['dates'] = dates_form_factor
    out['form_factors'] = form_factors
    return out

In [85]:
form_fac = form_factor_calc(tide_df)

/var/folders/zg/jhrvqkbd33x0c7lyzw9rdfsc0000gn/T/ipykernel_62952/3061471016.py:62: OptimizeWarning: Covariance of the parameters could not be estimated
  popt, pcov = scipy.optimize.curve_fit(sines, seconds_tide, tide_window, p0=initial_guess)


In [87]:
form_fac.head()

,dates,form_factors
0,2011-01-01 09:00:00,2.981448
1,2011-01-01 15:00:00,2.941475
2,2011-01-01 21:00:00,2.761705
3,2011-01-02 03:00:00,2.980646
4,2011-01-02 09:00:00,3.012552
